# Phase 247: Localization geometry control at ESM2-3B

Phase 236 (35M): topology's classification-side residual over geometry (+0.0097 PR-AUC) was confirmed only on
PR-AUC; the analogous localization residual (geometry+topology vs geometry, pooled DCC@4A) was NULL per seed
(+0.0051, 3/8 seeds) but trended positive with ensemble size (Phase 237). Phases 238-241 then found the
CLASSIFICATION residual becomes clearly confirmed (both PR-AUC and ROC-AUC) at 650M and 3B. This asks whether
the LOCALIZATION residual does the same, using ESM2-3B training data (Phase 239)
and a losslessly re-embedded ESM2-3B benchmark cache (Phase 245, decoded
from Phase 15's one-hot block, no re-fetch).

Protocol identical to Phase 236: training pool = the scale's own 2,055-chain blob, 92/8 split (RandomState 42),
`LocalizingEGNN` (lambda 0.1, 40 epochs, patience 8, batch 16), 4 arms x 8 seeds = 32 trainings, scored on the
1,149-unit COACH420/HOLO4K benchmark, pooled DCC@4A. **Primary comparison (fixed before any result): `geom+topo`
vs `geom`, seed-paired pooled DCC.**

## Cell 1: Install

In [ ]:
!pip install -q torch-geometric==2.6.1 biopython==1.84

## Cell 2: Load the ESM2-3B training graphs and geometry (Phase 229, scale-independent)

In [ ]:
import pickle

import gzip, json, os, time, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase239_graphs_esm3b.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"]); POS_WEIGHT = float(blob["pos_weight"])

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

splits = {s: [d for d in dataset if d.split == s] for s in ("train","val","test")}
print(f"device {DEVICE} | node dim {dataset[0].x.size(1)} | " +
      " | ".join(f"{k} {len(v)}" for k,v in splits.items()))

geo_tr = torch.load(locate("phase229_geometry_features.pt"), map_location="cpu", weights_only=False)
GEOM_DIM = len(geo_tr["names"]); assert GEOM_DIM == 17
SIDE_DIM = TOPO_DIM + GEOM_DIM
g_mu = torch.from_numpy(np.asarray(geo_tr["mean"], np.float32)); g_sd = torch.from_numpy(np.asarray(geo_tr["std"], np.float32))
_all = dataset; dataset = []; _dropped = 0
for d in _all:
    k = (str(d.pdb_id), str(d.chain_id))
    if k not in geo_tr["feats"]:
        _dropped += 1; continue
    f = geo_tr["feats"][k]; assert f.shape == (d.x.size(0), GEOM_DIM)
    z = ((torch.from_numpy(f) - g_mu) / g_sd).clamp(-5, 5)      # label-free statistics
    d.x = torch.cat([d.x, z], dim=1); dataset.append(d)
print(f"training chains with geometry: {len(dataset):,} (dropped {_dropped}); node dim {dataset[0].x.size(1)} = {N_PRE} + {TOPO_DIM} + {GEOM_DIM}")

## Cell 3: Recover true ligand centers for the training set and split 92/8 (Phase 22, verbatim)

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_LIG_ATOMS = 6
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

need = sorted({d.pdb_id for d in dataset})
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=16) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

def ligands_and_ca(pid, path, chain_id, n_expect):
    from Bio.PDB import MMCIFParser
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path,"rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[chain_id]
    except Exception:
        return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues) != n_expect: return None
    raw_ca = np.asarray([r["CA"].coord for r in residues], dtype=np.float32)

    ligs = []
    for c2 in model:
        for res in c2:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                             dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS: ligs.append(xyz)
    return raw_ca, ligs

t0=time.time(); n_ok=0; n_geom_fail=0; n_no_ligand=0
for d in dataset:
    if d.pdb_id not in paths: continue
    r = ligands_and_ca(d.pdb_id, paths[d.pdb_id], d.chain_id, int(d.num_nodes))
    if r is None: continue
    raw_ca, ligs = r
    if not ligs: n_no_ligand += 1; continue

    # geometry-drift check: pairwise CA distances must match d.pos (residue ORDER, not just count)
    k = min(50, len(raw_ca))
    a = np.linalg.norm(raw_ca[:k,None]-raw_ca[None,:k], axis=-1)
    b = np.linalg.norm(d.pos.numpy()[:k,None]-d.pos.numpy()[None,:k], axis=-1)
    if float(np.abs(a-b).max()) > 1e-2:
        n_geom_fail += 1; continue

    offset = raw_ca.mean(axis=0)                       # same centering d.pos already used
    y_np = d.y.numpy().astype(bool)
    contact_centroid = raw_ca[y_np].mean(axis=0) if y_np.any() else raw_ca.mean(axis=0)
    # "primary" ligand = the one whose center of mass is nearest the labelled contact residues
    coms = [lig.mean(axis=0) for lig in ligs]
    best = int(np.argmin([np.linalg.norm(c - contact_centroid) for c in coms]))
    d.true_center = torch.from_numpy((coms[best] - offset).astype(np.float32))
    n_ok += 1

print(f"\ntrue centers recovered: {n_ok:,}/{len(dataset):,} "
      f"({(time.time()-t0)/60:.1f} min) | geometry mismatches: {n_geom_fail} | no ligand: {n_no_ligand}")

dataset = [d for d in dataset if hasattr(d, "true_center")]
print(f"training chains with recovered true centers: {len(dataset):,}")

rng = np.random.RandomState(42)
perm = rng.permutation(len(dataset))
n_val = max(1, int(0.08 * len(perm)))
splits = {"val": [dataset[i] for i in perm[:n_val]], "train": [dataset[i] for i in perm[n_val:]]}
n_tr, n_va = len(splits["train"]), len(splits["val"])
print(f"deployment training: {n_tr} train / {n_va} val  (matches Phase 15/18's 2,056-chain setup)")

## Cell 4: Model (Phase 22, verbatim)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 5: Training machinery (Phase 22, verbatim except side-channel width)

In [ ]:
def run_epoch(model, loader, opt=None, lam=0.0):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    huber = nn.SmoothL1Loss(beta=4.0)                    # ~matches the 4A DCC threshold
    ps, ys, cd = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            logits, pred_c = model(b)
            loss = crit(logits, b.y)
            if lam > 0 and pred_c is not None:
                true_c = b.true_center.view(-1, 3)
                loss = loss + lam * huber(pred_c, true_c)
                cd.append((pred_c - true_c).norm(dim=1).detach().cpu().numpy())
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    out = {"pr": average_precision_score(y, p), "roc": roc_auc_score(y, p)}
    if cd:
        d_ = np.concatenate(cd)
        out["dcc"] = float((d_ < 4.0).mean()); out["mean_center_err"] = float(d_.mean())
    return out


def fit(use_loc_head, lam, seed, epochs=40, patience=8, batch=16):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LocalizingEGNN(N_PRE, SIDE_DIM, use_loc_head=use_loc_head).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(splits["train"], batch_size=batch, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=batch)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        run_epoch(model, tl, opt, lam)
        v = run_epoch(model, vl, lam=lam)
        sch.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], {k:t.clone() for k,t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    return model, best

## Cell 6: Arms; train 4 arms x 8 seeds

In [ ]:
import time
WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
ARMS = ["no-topo", "full", "geom", "geom+topo"]
TOPO_COLS = list(range(N_PRE, N_PRE + TOPO_DIM))
GEOM_COLS = list(range(N_PRE + TOPO_DIM, N_PRE + SIDE_DIM))
KEEP = {"no-topo": [], "full": TOPO_COLS, "geom": GEOM_COLS, "geom+topo": TOPO_COLS + GEOM_COLS}
ORIG_X = [d.x.clone() for d in dataset]
def apply_arm(a):
    for d, x0 in zip(dataset, ORIG_X):
        x = x0.clone(); mask = torch.ones(x.size(1), dtype=torch.bool)
        mask[N_PRE:] = False; mask[KEEP[a]] = True
        x[:, ~mask] = 0.0
        d.x = x

BEST_LAMBDA = 0.1
models = {a: {} for a in ARMS}; train_log = []
t0 = time.time()
for arm in ARMS:
    apply_arm(arm)
    for seed in range(8):
        m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=seed)
        v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
        models[arm][seed] = m
        torch.save(m.state_dict(), WORK/f"cache/loc247_{arm.replace('+','_')}_seed{seed}.pt")
        train_log.append({"arm": arm, "seed": seed, "val_pr": float(val_pr), "val_dcc": float(v.get("dcc", float("nan")))})
        print(f"  [{arm:9s}] seed {seed}: val PR {val_pr:.4f}  val DCC {v.get('dcc',float('nan')):.4f}  [{(time.time()-t0)/60:.1f}m]")
json.dump(train_log, open(WORK/"phase247_train_log.json", "w"), indent=2)

## Cell 7: Load the re-embedded ESM2-3B benchmark cache (Phase 245, saved as two shards by benchmark source to keep the save step's memory down) and its geometry (Phase 235, scale-independent)

In [ ]:
raw_cache = {}
n_pre_bench = None
for src in ("coach420", "holo4k"):
    shard = torch.load(locate("phase245_raw_features_esm3b." + src + ".pt"), map_location="cpu", weights_only=False)
    n_pre_bench = shard["n_pre"] if n_pre_bench is None else n_pre_bench
    assert shard["n_pre"] == n_pre_bench, "shard n_pre mismatch between coach420 and holo4k"
    for k, e in shard["cache"].items():
        e["x"] = e["x"].astype(np.float32)   # stored fp16 for the save step; cast back up for scoring
        raw_cache[k] = e
geo_bm = torch.load(locate("phase235_benchmark_geometry.pt"), map_location="cpu", weights_only=False)
blob6b_bench = torch.load(locate("phase6b_graphs.pt"), map_location="cpu", weights_only=False)
mu_b, sd_b = blob6b_bench["topo_mean"], blob6b_bench["topo_std"]
del blob6b_bench
valid_keys = {k for k in raw_cache if k in geo_bm["feats"]}
print(f"benchmark chains: {len(raw_cache):,} cached (ESM2-3B), {len(valid_keys):,} with geometry (dropped {len(raw_cache)-len(valid_keys)} for every arm)")
assert n_pre_bench == N_PRE, f"benchmark N_PRE {n_pre_bench} != training N_PRE {N_PRE}"

## Cell 8: Score all arms and seeds (Phase 22/236 scoring, arm masks applied to the benchmark features)

In [ ]:
from collections import defaultdict

def bench_x(entry, key, arm):
    x = torch.from_numpy(entry["x"]).clone()
    topo = (x[:, N_PRE:] - torch.from_numpy(mu_b)) / torch.from_numpy(sd_b)
    g = ((torch.from_numpy(geo_bm["feats"][key]) - g_mu) / g_sd).clamp(-5, 5)
    x = torch.cat([x[:, :N_PRE], topo, g], dim=1)
    mask = torch.ones(x.size(1), dtype=torch.bool); mask[N_PRE:] = False; mask[KEEP[arm]] = True
    x[:, ~mask] = 0.0
    return x

@torch.no_grad()
def predict_all_seeds(entry, key, arm):
    d = type("D", (), {})()
    d.x = bench_x(entry, key, arm).to(DEVICE); d.pos = torch.from_numpy(entry["pos"]).to(DEVICE)
    d.edge_index = torch.from_numpy(entry["edge_index"]).to(DEVICE); d.edge_attr = torch.from_numpy(entry["edge_attr"]).to(DEVICE)
    d.batch = torch.zeros(d.x.size(0), dtype=torch.long, device=DEVICE)
    offset = entry["ca"].mean(axis=0); out = {}
    for seed, m in models[arm].items():
        lg, pc = m(d)
        out[seed] = {"prob_sum": float(torch.sigmoid(lg).sum().cpu()), "center": pc.cpu().numpy()[0] + offset}
    return out

def dcc_dist(pred_center, ligs):
    if pred_center is None or not ligs: return None
    best = np.inf
    for lig_atoms in ligs:
        if len(lig_atoms) == 0: continue
        best = min(best, float(np.linalg.norm(pred_center - lig_atoms.mean(axis=0))))
    return best if np.isfinite(best) else None

t0 = time.time()
by_structure = defaultdict(list)
for key in valid_keys: by_structure[key[:2]].append((key[2], key))
out = {a: {} for a in ARMS}; ligs_of = {}
for skey, chain_list in by_structure.items():
    ligs_of[skey] = raw_cache[chain_list[0][1]]["ligs"]
    for arm in ARMS:
        out[arm][skey] = [(cid, predict_all_seeds(raw_cache[k], k, arm)) for cid, k in chain_list]
print(f"inference (8 seeds x {len(ARMS)} arms x {len(by_structure)} structures) in {(time.time()-t0)/60:.1f} min")

def hits_for(arm, seeds, src=None):
    """dict structure -> 0/1 hit, ensemble over `seeds`, best chain by mean summed probability."""
    res = {}
    for skey, chain_list in out[arm].items():
        if src is not None and skey[0] != src: continue
        best_conf, best_center = -np.inf, None
        for cid, per_seed in chain_list:
            cs = [per_seed[s]["center"] for s in seeds if s in per_seed]; fs = [per_seed[s]["prob_sum"] for s in seeds if s in per_seed]
            if not cs: continue
            c, f = np.mean(cs, axis=0), np.mean(fs)
            if f > best_conf: best_conf, best_center = f, c
        dd = dcc_dist(best_center, ligs_of[skey])
        if dd is None: continue
        res[skey] = int(dd < 4.0)
    return res

def rate(h): return float(np.mean(list(h.values()))) if h else float("nan")

## Cell 9: Per-seed pooled DCC, seed-paired statistics (primary comparison first), structure-level bootstrap on the 8-seed ensembles

In [ ]:
from scipy.stats import ttest_rel, wilcoxon
per_seed = {a: np.array([rate(hits_for(a, (s,))) for s in range(8)]) for a in ARMS}
per_seed_bm = {(a, src): np.array([rate(hits_for(a, (s,), src)) for s in range(8)]) for a in ARMS for src in ("coach420", "holo4k")}
print(f"{'arm':<10}{'pooled DCC mean':>17}{'std':>8}   {'COACH420':>9}{'HOLO4K':>9}")
for a in ARMS:
    print(f"{a:<10}{per_seed[a].mean():>17.4f}{per_seed[a].std():>8.4f}   {per_seed_bm[(a,'coach420')].mean():>9.4f}{per_seed_bm[(a,'holo4k')].mean():>9.4f}")

COMPS = [("geom+topo", "geom"), ("full", "no-topo"), ("geom", "no-topo"), ("geom", "full"), ("geom+topo", "full")]
rng = np.random.default_rng(236)
ens = {a: hits_for(a, tuple(range(8))) for a in ARMS}
rows = []
print(f"\n{'comparison (a - b)':<26}{'per-seed delta':>15}{'wins':>6}{'t p':>9}{'W p':>9}   {'8-seed ensemble delta [95% bootstrap over structures]':>58}")
for a, b in COMPS:
    d = per_seed[a] - per_seed[b]
    tp, wp = ttest_rel(per_seed[a], per_seed[b]).pvalue, wilcoxon(d).pvalue
    keys = sorted(set(ens[a]) & set(ens[b])); ha = np.array([ens[a][k] for k in keys]); hb = np.array([ens[b][k] for k in keys])
    idx = rng.integers(0, len(keys), size=(5000, len(keys)))
    bs = (ha[idx].mean(1) - hb[idx].mean(1))
    lo, hi = np.percentile(bs, [2.5, 97.5]); dens = float(ha.mean() - hb.mean())
    rows.append(dict(a=a, b=b, mean_seed_delta=float(d.mean()), wins=int((d > 0).sum()), t_p=float(tp), wilcoxon_p=float(wp),
                     ens_delta=dens, ens_ci_lo=float(lo), ens_ci_hi=float(hi), n_structures=len(keys)))
    print(f"{a+' - '+b:<26}{d.mean():>+15.4f}{int((d>0).sum()):>4}/8{tp:>9.1e}{wp:>9.1e}   {dens:>+8.4f} [{lo:+.4f}, {hi:+.4f}]  n={len(keys)}")
print(f"\nPhase 22 reference (old checkpoints, no geometry arm): full - no_topo = +0.0528 pooled DCC")
import pandas as pd
pd.DataFrame(rows).to_csv(WORK/"phase247_paired_stats.csv", index=False)
json.dump({"per_seed": {a: per_seed[a].tolist() for a in ARMS},
           "per_seed_by_benchmark": {f"{a}|{s}": per_seed_bm[(a, s)].tolist() for (a, s) in per_seed_bm},
           "ensemble_rate": {a: rate(ens[a]) for a in ARMS}, "n_structures": len(by_structure),
           "n_bench_chains_valid": len(valid_keys), "n_bench_chains_total": len(raw_cache), "n_train_chains": len(dataset)},
          open(WORK/"phase247_summary.json", "w"), indent=2)
print("saved phase247_paired_stats.csv, phase247_summary.json, phase247_train_log.json")